In [10]:
import json
import time
import requests
from typing import Dict, Any, Iterable
import os
from pathlib import Path

In [11]:
STACKEXCHANGE_BASE = "https://api.stackexchange.com/2.3"
SITE = "math"
PAGE_SIZE = 100
MAX_PAGE = 25
FILTER_WITH_BODY = "!9_bDDxJY5"
# Determine notebook directory: prefer __file__ when available, otherwise use cwd
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
# OUTPUT_FILE as an absolute, normalized path
OUTPUT_FILE = str((NOTEBOOK_DIR / ".." / "data" / "questions.jsonl").resolve())

In [12]:
def fetch_page(page: int) -> Dict[str, Any]:
    params = {
        "site": SITE,
        "page": page,
        "pagesize": PAGE_SIZE,
        "order": "asc",
        "sort": "creation",
        "filter": FILTER_WITH_BODY,
    }
    try:
        r = requests.get(f"{STACKEXCHANGE_BASE}/questions", params=params, timeout=30)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.HTTPError as e:
        status = getattr(e.response, "status_code", None)
        if status == 400:
            print(f"Received 400 Bad Request for page {page}; treating as end of results.")
            return {"items": [], "has_more": False}
        print(f"HTTP Error on page {page}: {e}")
        raise

In [13]:
def iter_questions() -> Iterable[Dict[str, Any]]:
    page = 1

    while True:
        if page > MAX_PAGE:
            print(f"Reached configured max page ({MAX_PAGE}). Stopping.")
            break

        data = fetch_page(page)

        for q in data.get("items", []):
            yield q

        if "backoff" in data:
            time.sleep(data["backoff"])

        if not data.get("has_more", False):
            break

        page += 1

In [14]:
def main() -> None:
    count = 0

    with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
        for question in iter_questions():
            f.write(json.dumps(question, ensure_ascii=False) + "\n")
            count += 1

            if count % 1000 == 0:
                print(f"Fetched {count} questions")

    print(f"Done. Total questions written: {count}")

In [15]:
if __name__ == "__main__":
    main()

Fetched 1000 questions
Fetched 2000 questions
Fetched 2000 questions
Reached configured max page (25). Stopping.
Done. Total questions written: 2500
Reached configured max page (25). Stopping.
Done. Total questions written: 2500
